# Development Plans Dataset — Kalomo Town Council

**Owner:** Josiphiah (Issue #4) &nbsp;|&nbsp; **CSC4792 Mini Project, Group project — Kalomo Town Council, Zambia**

This notebook documents how `db-unza26-csc4792-kalomo_town_council_development_plans.csv` was built: which pages of [kalomocouncil.gov.zm](https://www.kalomocouncil.gov.zm/) were scraped, how the source PDFs were found and processed, and every decision made while turning unstructured planning documents into a clean, structured table.

The two supporting scripts referenced throughout are:
- `scripts/scraping/scrape_development_plans.py`
- `scripts/cleaning/clean_development_plans.py`

Both were run from the command line to produce the files this notebook loads and inspects below; re-running the code cells here does **not** re-scrape the live site (see Step 2), so this notebook works offline once the repo is cloned.

## Step 1 — Manually browsing the site first

Before writing any scraping code, the council site's menu was browsed by hand to find where planning documents actually live. The site (a WordPress/Elementor build) does **not** have a single "Development Plans" menu item — instead, relevant documents are split across two pages:

1. **ZDSP** (`?page_id=3889`) — a "KEY DOCUMENTS" list including the National Decentralisation Policy, ESMPs for specific projects, and a ZDSP proposed-projects list. "ZDSP" stands for **Zambia Devolution Support Programme**, confirmed from the Citizen Engagement Strategy PDF's own cover page (Ministry of Local Government and Rural Development / Zambia Devolution Support Programme, 2025).
2. **Publications** (`?page_id=4451`) — a tabbed page (all tab content is present in the raw HTML at once, just hidden/shown by JavaScript, so a plain `requests.get` still sees every tab). The tabs relevant to this dataset are **IDP**, **Investment Profile**, **Procurement Plan**, and **Engagement Plan**. Other tabs on the same page (Financial Statements, Minutes, Acts and Policies) were left to Faiz's and Nicholas's datasets.

This manual pass is what produced the `SOURCE_PAGES` list and the `RELEVANT_KEYWORDS` filter in the scraping script below — both were chosen from what was actually found on the page, not guessed in advance.

## Step 2 — Automated scraping

`scripts/scraping/scrape_development_plans.py` fetches both source pages, pulls every `<a href="...pdf">` link out of the HTML with BeautifulSoup, filters to the ones relevant to development plans (by keyword match on link text/URL), downloads each PDF into `data/raw/development_plans/pdfs/`, and extracts text with `pdfplumber` into a companion `..._extract.txt` file for manual review.

The cell below imports the script as a module (without calling `main()`, so no network requests are made here) just to show the exact source pages and filter keywords it used — the real run was done from the command line with `python scripts/scraping/scrape_development_plans.py`.

In [1]:
import sys, os
sys.path.append(os.path.join("..", "scripts", "scraping"))
import scrape_development_plans as sdp

print("Source pages manually identified and scraped:")
for p in sdp.SOURCE_PAGES:
    print(" -", p)

print("\nKeywords used to filter PDF links to development-plan-relevant documents:")
print(sdp.RELEVANT_KEYWORDS)

Source pages manually identified and scraped:
 - https://www.kalomocouncil.gov.zm/?page_id=3889
 - https://www.kalomocouncil.gov.zm/?page_id=4451

Keywords used to filter PDF links to development-plan-relevant documents:
['idp', 'investment', 'zdsp', 'decentralisation', 'procurement', 'esmp', 'debt-arrears', 'citizen_engagement', 'stakeholder-engagement']


## Step 3 — Problems encountered while scraping

**TLS certificate issue.** A plain `requests.get()` against `https://www.kalomocouncil.gov.zm` fails with `SSLCertVerificationError: unable to get local issuer certificate`. This was confirmed independently with `curl -v` — the council's server sends an incomplete certificate chain. The site is a legitimate, publicly reachable government site (it opens fine in a browser, which fills the chain gap using its own trust store), so `verify=False` is used specifically for this host, with `urllib3`'s resulting warning silenced. No credentials are ever sent over these requests, so this only reduces tamper-detection on public, unauthenticated GET requests to PDFs and public pages.

**A keyword filter that was initially too broad.** The first version of the relevant-document filter included a plain `"engagement"` keyword, which also matched several council **meeting-minutes** PDFs (e.g. *"2025 EXTRACT MINUTES-STAKEHOLDERS ENGAGEMENT PLAN"*, *"2025 BUSINESS ENGAGEMENT MINUTES-BUDGET"*) — governance/financial records, not development plans, and out of scope for this dataset (they belong with Nicholas's or Faiz's data instead). The filter was tightened to two more specific keywords (`"citizen_engagement"`, `"stakeholder-engagement"`) that match only the actual strategy/plan documents by their exact filename pattern, and the scrape was re-run. This is exactly why the filter keywords were reviewed by re-running the script rather than trusted on the first pass — the code cell below reads the *current* (corrected) keyword list.

**Scanned/image PDFs.** Of the 10 relevant PDFs (after the filter fix above), 4 have no *meaningful* extractable text — `pdfplumber` returns only a few bytes of blank-page artifacts (stray carriage-return characters, no real words) even though the file downloads correctly:

- `2025-STAKEHOLDER-ENGAGEMENT-PLAN.pdf` (2 bytes extracted)
- `ZDSP-PROPOSED-PROJECTS_rotated.pdf` (0 bytes extracted)
- `ESMP-TRUCK-PARKING-BAY.pdf` (38 bytes, all blank-line artifacts)
- `ESMP-TANDABALE-MARKET-SHELTER.pdf` (38 bytes, all blank-line artifacts)

These are scanned photographs/images of paper documents rather than text-based PDFs. No OCR step was added (out of scope given the project timeline), so these four are still included in the final dataset as rows, but their `description` field says explicitly that the source has no extractable text rather than inventing detail that can't be verified from the document itself.

In [2]:
raw_dir = os.path.join("..", "data", "raw", "development_plans")
for fname in sorted(os.listdir(raw_dir)):
    if fname.endswith("_extract.txt"):
        path = os.path.join(raw_dir, fname)
        with open(path, encoding="utf-8") as f:
            content = f.read()
        meaningful_chars = len(content.strip())
        flag = "NO MEANINGFUL TEXT (blank-page artifacts only)" if meaningful_chars < 50 else f"{meaningful_chars} chars extracted"
        print(f"{fname:65s} {flag}")

2025-STAKEHOLDER-ENGAGEMENT-PLAN_extract.txt                      NO MEANINGFUL TEXT (blank-page artifacts only)
Citizen_Engagement_Strategy_extract.txt                           11337 chars extracted
Debt-Arrears-Monitoring-Mechanism-FINAL-COPY8-min_extract.txt     26235 chars extracted
ESMP-TANDABALE-MARKET-SHELTER_extract.txt                         NO MEANINGFUL TEXT (blank-page artifacts only)
ESMP-TRUCK-PARKING-BAY_extract.txt                                NO MEANINGFUL TEXT (blank-page artifacts only)
KALOMO-IDP-FINAL-JANUARY-2023-LAUNCH-1-1-2_extract.txt            24008 chars extracted
Kalomo-town-Council-2.0-Investment-Profile-1-1_extract.txt        20888 chars extracted
National-Decentralisation-Policy-2023_extract.txt                 35819 chars extracted
ZDSP-PROPOSED-PROJECTS_rotated_extract.txt                        NO MEANINGFUL TEXT (blank-page artifacts only)
plan_extract.txt                                                  18323 chars extracted


## Step 4 — Structuring unstructured documents into rows

Almost none of these source documents are tables — they are prose planning/policy PDFs. Following the same approach used for the CDF dataset, each `_extract.txt` file was read manually, and the plan/project name, sector, period, status and a short description were pulled out by hand into `clean_development_plans.py`. Two kinds of row were built:

1. **Document-level rows** — one row per distinct planning/policy document (the IDP itself, the Investment Profile, the National Decentralisation Policy, etc). 10 of these.
2. **Project-level rows** — the IDP contains its own **Table 28: Capital Investment Plan**, listing 10 specific proposed projects (e.g. "Construction of a Truck Yard", "Construction of Dams"). These were pulled out individually with `pdfplumber.extract_tables()` rather than left as one "IDP" row, so the dataset captures actual named projects and not just document titles.

**A genuine data quality finding, not a bug:** `extract_tables()` on IDP page 152 confirms that the council's own "Amount (ZMW)" and "S/N" columns in Table 28 are blank in the *source document itself* — the council published this table without filling in budgeted amounts. Rather than inventing figures, every one of these 10 rows is marked `"Planned (amount not specified in source IDP)"` in the `status` column, except "Construction of a Truck Yard", which the 2025 Procurement Plan separately confirms is under active implementation (cross-referenced as *"Completion of a Truck Yard wall fence", KTC/2025/2*).

**Standardisation applied** in `clean_development_plans.py`, matching the conventions agreed for every dataset in this project:
- `record_id`: assigned sequentially as `DP-001` … `DP-020`
- Every text field stripped of whitespace; empty/`nan`/`None` values normalised to the single literal string `"N/A"` (never a mix of blanks, dashes, or "unknown")
- Duplicate rows dropped on `(plan_name, source_url)`
- Every row asserted to have a working `https://` `source_url` and a unique `record_id` before the file is written
- `date_scraped` stamped with the date the cleaning script was run
- Output written with `sep="|"` per the project's naming/format convention

## Step 5 — Loading and inspecting the final dataset

In [3]:
import pandas as pd

csv_path = os.path.join("..", "data", "processed", "db-unza26-csc4792-kalomo_town_council_development_plans.csv")
df = pd.read_csv(csv_path, sep="|", keep_default_na=False)
df.head(10)

,record_id,plan_name,plan_type,sector,period,description,status,source_url,date_scraped
0,DP-001,Kalomo District Integrated Development Plan (I...,IDP,Multi-sector,2021-2030,District-wide Integrated Development Plan prep...,Adopted,https://www.kalomocouncil.gov.zm/wp-content/up...,2026-09-11
1,DP-002,Kalomo Town Council 2.0 Investment Profile,Investment Profile,Investment/Economic Development,N/A,Investor-facing profile covering potential via...,Published,https://www.kalomocouncil.gov.zm/wp-content/up...,2026-09-11
2,DP-003,National Decentralisation Policy (2023),Policy,Governance/Decentralisation,2023,Revised national policy (Office of the Preside...,In force,https://www.kalomocouncil.gov.zm/wp-content/up...,2026-09-11
3,DP-004,Council Citizen Engagement Strategy (Output-Ba...,Strategy,Governance/Public Participation,2025,Ministry of Local Government and Rural Develop...,Draft/template (marked 'Official Use Only' in ...,https://www.kalomocouncil.gov.zm/wp-content/up...,2026-09-11
4,DP-005,Stakeholders Engagement Plan 2025,Plan,Governance/Public Participation,2025,Council's 2025 stakeholder engagement plan. So...,Published,https://www.kalomocouncil.gov.zm/wp-content/up...,2026-09-11
5,DP-006,Kalomo Town Council Procurement Plan 2025,Plan,Procurement,2025,"Annual procurement plan (version 1, last updat...",Active,https://www.kalomocouncil.gov.zm/wp-content/up...,2026-09-11
6,DP-007,The Local Authorities Debt and Arrears Monitor...,Mechanism/Policy,Finance,2023,"National-level (Republic of Zambia) mechanism,...",In force,https://www.kalomocouncil.gov.zm/wp-content/up...,2026-09-11
7,DP-008,ZDSP Proposed Projects list,Project List (ZDSP),Multi-sector,N/A,Council-submitted list of proposed projects un...,Proposed,https://www.kalomocouncil.gov.zm/wp-content/up...,2026-09-11
8,DP-009,Truck Parking Bay - Environmental and Social M...,ESMP,Infrastructure/Transport,N/A,Project-level environmental and social safegua...,Planned,https://www.kalomocouncil.gov.zm/wp-content/up...,2026-09-11
9,DP-010,Tandabale Market Shelter - Environmental and S...,ESMP,Infrastructure/Markets,N/A,Project-level environmental and social safegua...,Ongoing (per 2025 Procurement Plan),https://www.kalomocouncil.gov.zm/wp-content/up...,2026-09-11


In [4]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 20 entries, 0 to 19
Data columns (total 9 columns):
 #   Column        Non-Null Count  Dtype
---  ------        --------------  -----
 0   record_id     20 non-null     str  
 1   plan_name     20 non-null     str  
 2   plan_type     20 non-null     str  
 3   sector        20 non-null     str  
 4   period        20 non-null     str  
 5   description   20 non-null     str  
 6   status        20 non-null     str  
 7   source_url    20 non-null     str  
 8   date_scraped  20 non-null     str  
dtypes: str(9)
memory usage: 1.5 KB


In [5]:
df.describe(include="all")

,record_id,plan_name,plan_type,sector,period,description,status,source_url,date_scraped
count,20,20,20,20,20,20,20,20,20
unique,20,20,9,18,4,11,10,10,1
top,DP-001,Kalomo District Integrated Development Plan (I...,Capital Investment Project (IDP Table 28),Multi-sector,2021-2030,Project listed in the IDP's Capital Investment...,Planned (amount not specified in source IDP),https://www.kalomocouncil.gov.zm/wp-content/up...,2026-09-11
freq,1,1,10,2,11,10,9,11,20


In [6]:
print("Rows by plan_type:")
print(df["plan_type"].value_counts())
print("\nRows by status:")
print(df["status"].value_counts())
print("\nRows with period = N/A:", (df["period"] == "N/A").sum(), "out of", len(df))

Rows by plan_type:
plan_type
Capital Investment Project (IDP Table 28)    10
Plan                                          2
ESMP                                          2
IDP                                           1
Investment Profile                            1
Policy                                        1
Strategy                                      1
Mechanism/Policy                              1
Project List (ZDSP)                           1
Name: count, dtype: int64

Rows by status:
status
Planned (amount not specified in source IDP)                                                                       9
Published                                                                                                          2
In force                                                                                                           2
Adopted                                                                                                            1
Draft/template (marke

## Limitations

- 4 of 20 rows are drawn from scanned/image PDFs with no extractable text; their descriptions are limited to what could be inferred from the page/menu context they were listed under, not full document content.
- Sector labels on the 10 IDP Capital Investment Plan rows (e.g. "Dip Tanks → Agriculture/Livestock") are inferred from context by the person compiling this dataset, since the source table itself only lists a project name and a blank amount column — they are not verbatim classifications from the council.
- No amounts (ZMW) are available for the 10 Capital Investment Plan projects because the source IDP itself does not publish them; this is a genuine gap in the council's own published document, not a scraping failure.
- Raw PDFs (~140MB combined) are intentionally **not** committed to this repository (see `.gitignore`) to keep the repo lightweight for cloning; every row's `source_url` points to where the original document can be re-downloaded from the council's site.

See `docs/DATA_DICTIONARY.md` for the full column reference for this and every other dataset in this project.